In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
import os
from openai import OpenAI

from dotenv import load_dotenv
import os

load_dotenv("youtube-learning-assistant/.env", override=True)

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OpenRouter API key not found!")

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 Indexing 

In [4]:
from urllib.parse import urlparse, parse_qs

def extract_video_id(video_url):

    parsed_url = urlparse(video_url.strip())

    hostname = parsed_url.hostname or ""

    video_id = None

    if hostname in ["youtu.be", "www.youtu.be"]:
        video_id = parsed_url.path.lstrip("/").split("/")[0]

    elif hostname in [
        "youtube.com",
        "www.youtube.com",
        "m.youtube.com",
        "music.youtube.com"
    ]:
        if parsed_url.path == "/watch":
            video_id = parse_qs(parsed_url.query).get("v", [None])[0]

        elif parsed_url.path.startswith(("/shorts/", "/embed/", "/live/")):
            video_id = parsed_url.path.split("/")[2]

    if not video_id or not video_id.replace("-", "").replace("_", "").isalnum():
        raise ValueError("Invalid YouTube URL. Please provide a valid video URL.")

    if len(video_id) != 11:
        raise ValueError("Invalid YouTube video ID.")

    return video_id

In [5]:
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable
)

# Accept YouTube URL from user
video_url = input("Paste your YouTube Video URL: ")

# Extract video ID automatically
video_id = extract_video_id(video_url)

print("Extracted Video ID:", video_id)

try:
    # Get available transcripts
    transcript_list = YouTubeTranscriptApi().list(video_id)

    # Prefer Hindi, then English
    try:
        selected_transcript = transcript_list.find_transcript(["hi", "en"])
    except NoTranscriptFound:
        selected_transcript = next(iter(transcript_list))

    # Fetch transcript
    transcript_data = selected_transcript.fetch()

    transcript = " ".join(
        snippet.text for snippet in transcript_data
    )

    print("Source Language:", selected_transcript.language_code)
    print("Transcript extracted successfully!")
    print("Total characters:", len(transcript))

except TranscriptsDisabled:
    print("Captions are disabled for this video.")

except NoTranscriptFound:
    print("No transcript available.")

except VideoUnavailable:
    print("Video is unavailable.")

Extracted Video ID: gm_lQG8fYjI
Source Language: hi
Transcript extracted successfully!
Total characters: 60925


In [6]:
transcript_list

## text spliter

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [8]:
len(chunks)

76

## embedding generation and storing in vector store (fiass)

In [9]:
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

vector_store = FAISS.from_documents(chunks, embeddings)

In [28]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# Dedicated translation LLM
translation_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=2000
)

# Function to translate one chunk
def translate_chunk(chunk):

    translation_prompt = f"""
    Translate the following transcript into English.

    Instructions:
    1. Translate the complete content.
    2. Do not summarize or omit anything.
    3. Preserve technical terminology.
    4. Maintain the original meaning and sequence.
    5. Return only the translated text.

    Transcript:
    {chunk.page_content}
    """

    response = translation_llm.invoke(translation_prompt)

    return response.content


# Parallel processing configuration
MAX_WORKERS = 8

# Store results in their original order
translated_chunks = [None] * len(chunks)

print(f"Total chunks: {len(chunks)}")
print(f"Parallel workers: {MAX_WORKERS}")

# Execute translation concurrently
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {
        executor.submit(translate_chunk, chunk): i
        for i, chunk in enumerate(chunks)
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Translating transcript"
    ):

        index = futures[future]

        translated_chunks[index] = future.result()


# Combine translated chunks in original order
english_transcript = "\n\n".join(translated_chunks)

print("\nTranslation completed!")
print("Total translated chunks:", len(translated_chunks))
print("Total characters:", len(english_transcript))

print(english_transcript)

Total chunks: 76
Parallel workers: 8


Translating transcript:   0%|          | 0/76 [00:00<?, ?it/s]

Translating transcript: 100%|██████████| 76/76 [00:38<00:00,  2.00it/s]


Translation completed!
Total translated chunks: 76
Total characters: 77962
Hi guys, my name is Nitish and you are welcome to my YouTube channel. In this video, we will continue our LangChain playlist. This video is the 18th and last video of this playlist as of now. It is possible that I may add some more videos here in the future. But according to my initial plan, which I shared with you in the first video, today’s video is going to be the last video. Now let me tell you what we are going to cover in today’s video. We will cover how we can create AI agents with the help of LangChain. Alright? So in the last two videos, we learned about tools and the concept of tool calling, which we will use in today’s video. And today’s video is mostly divided into two parts. In the first part, I will give you a little theoretical and conceptual introduction about what AI agents are. We won’t go too deep; we will discuss briefly. But I would like to explain to you what AI agents are through a use ca

In [29]:
vector_store.index_to_docstore_id

{0: 'f3402349-6ae3-4f0e-aa24-9608aec068db',
 1: '5ed2ddf7-06ee-47c6-a6ec-56594ad63fad',
 2: '621aea12-941d-44a7-bf4e-cca56140a14d',
 3: '087a60b3-804c-4548-bda4-8e787ff5667c',
 4: '77e8d05a-d1e6-43b3-af14-30e208480408',
 5: 'c7e05591-6070-4261-bb6e-c737f48cf1c4',
 6: '5edaa8f9-69e9-4aab-96f2-da8201e201a7',
 7: '1ebf1f37-bfce-424f-8360-441ce549619f',
 8: '509089fa-ac70-4c5c-a361-a0ff63bc12b8',
 9: '7b8c8082-0c10-44db-accb-b33791fb95f6',
 10: '78b50be8-d95b-4b57-bdf9-3629b9b048c3',
 11: 'b74e0ec3-24de-4c95-8475-9b466f2ff93e',
 12: '50ff5572-7342-4e2f-b4a0-aa92a84d4a32',
 13: '64ec664f-d22c-4eae-b283-6b0f6e925ac5',
 14: '301423c7-d7c2-4f0e-908e-82c96c5658be',
 15: '9aedb8ee-6f48-45ae-abe9-81b8f759c7b6',
 16: '7ada6f8d-d35e-4f8d-bf63-e7a1d386175b',
 17: 'abce4eb9-38ed-43b6-bc04-706aa9ccd1f0',
 18: '9ab06b5a-1df4-41c6-a7fc-49c1706ea841',
 19: 'd1bab008-f5e7-431d-808e-c52afc272998',
 20: '92ebbb12-6af9-460b-ab4f-18508d91f5a7',
 21: '25e84e91-0565-42e5-9e0c-1491d70f2754',
 22: '1dd71113-908f-

## retrieval

In [30]:
retrieval = vector_store.as_retriever(search_type = "similarity", search_kwargs={"k": 3})

In [31]:
retrieval.invoke("what is neural networks?")

[Document(id='3e686faa-1efc-4bf3-a5c4-e18fe3b17f9c', metadata={}, page_content='वर्किंग? तो शुड आई सजेस्ट यू बस का ट्रैवल और वह इस तरीके से रीअप्ट कर जाता है। ठीक है? तो दीज़ आर द मेन कैरेक्टरिस्टिक्स ऑफ एन एआई एजेंट। नटशेल में इट्स एन इंटेलिजेंट सिस्टम जो किसी भी गिवन गोल को खुद ब खुद एग्जीक्यूट करके आपको दे देगा। और यह करने के लिए वो दो चीजों का इस्तेमाल करता है। एक एलएलएम जिसकी हेल्प से वो डिसीजन मेकिंग करता है, प्लानिंग करता है। सेकंड टूल्स जिसकी हेल्प से वो सारे एक्शंस परफॉर्म करता है। एंड दीज़ आर द एस्पेक्ट्स दैट एन एआई एजेंट हैज़। ठीक है? तो नटशेल में मैंने आपको यह भी बता दिया कि एआई एजेंट क्या इंप्रूवमेंट ला रहे हैं हमारे एकिस्टिंग सिस्टम्स के ऊपर। मैंने आपको यह भी बता दिया कि एआई एजेंट्स टेक्निकली स्पीकिंग क्या होते हैं और उनकी कोर कैरेक्टरिस्टिक्स क्या होती हैं। तो ये ओवरव्यू मुझे देना था इस वीडियो में। वो मैंने आपको दे दिया। अब हम एक बहुत इंटरेस्टिंग काम करेंगे। हम लैंगचेन की हेल्प से एक बेसिक एआई एजेंट बिल्ड करेंगे। ठीक है? सो लेट्स मूव ऑन टू दैट। सो गाइस मैंने प्लान किया है क

## augmentation

In [32]:
llm = ChatOpenAI(model="openai/gpt-4o-mini", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0.7, max_tokens=500)

In [33]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant that answers questions based on the context provided.
    Answer only from the provided context. If the context is insufficient, say "I don't know".

    Context:
    {context}

    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [34]:
question = "Is Neural Network is discussed in this video? if yes, then what was discussed about  in this video?"
retrieved_docs = retrieval.invoke(question)

In [35]:
retrieved_docs

[Document(id='6c1ea135-3c5f-4fb8-9eed-41320b14ab5b', metadata={}, page_content='हेल्प करती हैं। बट यहां पे मैंने क्यों पढ़ाया? बिकॉज़ मैं आपको दिखाना चाहता था कि लैंग चेन में भी यह करना पॉसिबल है। करेंटली दिस इज पॉसिबल। आपको कोई रोक नहीं रहा है। इट्स जस्ट कि इट्स नॉट रिकमेंडेड कि आप लैंग चेन यूज़ करो। तो मैं बस आपको दिखाना चाहता था कि इट इज़ पॉसिबल टू क्रिएट एजेंट्स यूजिंग लंग चेन। एंड सेकंड थोड़ा सा आपको एक फ्लेवर देना था कि एआई एजेंट्स होते क्या है? बनाए कैसे जाते हैं? उनका वर्किंग कैसा होता है? यह सब कुछ आपको दिखाना चाहता था। तो आई रियली होप आपको यह सब कुछ देख के मजा आया। और ऑन द होल भी यह पूरी प्लेलिस्ट अगर आपने फॉलो की तो आई रियली होप लैंग चेंज जो कि अभी एक बहुत पॉपुलर लाइब्रेरी है। यह आप आसानी से समझ पाए। तो या इट हैज़ बीन अ ग्रेट जर्नी। थैंक यू अगर आपने सारे वीडियोस देखे और कमेंट किया, लाइक किया, शेयर किया। इस वीडियो को भी प्लीज लाइक करना अगर आपको पसंद आए। बाकी अगर आपने चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाई।'),
 Document(id='3e686faa-

In [36]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'हेल्प करती हैं। बट यहां पे मैंने क्यों पढ़ाया? बिकॉज़ मैं आपको दिखाना चाहता था कि लैंग चेन में भी यह करना पॉसिबल है। करेंटली दिस इज पॉसिबल। आपको कोई रोक नहीं रहा है। इट्स जस्ट कि इट्स नॉट रिकमेंडेड कि आप लैंग चेन यूज़ करो। तो मैं बस आपको दिखाना चाहता था कि इट इज़ पॉसिबल टू क्रिएट एजेंट्स यूजिंग लंग चेन। एंड सेकंड थोड़ा सा आपको एक फ्लेवर देना था कि एआई एजेंट्स होते क्या है? बनाए कैसे जाते हैं? उनका वर्किंग कैसा होता है? यह सब कुछ आपको दिखाना चाहता था। तो आई रियली होप आपको यह सब कुछ देख के मजा आया। और ऑन द होल भी यह पूरी प्लेलिस्ट अगर आपने फॉलो की तो आई रियली होप लैंग चेंज जो कि अभी एक बहुत पॉपुलर लाइब्रेरी है। यह आप आसानी से समझ पाए। तो या इट हैज़ बीन अ ग्रेट जर्नी। थैंक यू अगर आपने सारे वीडियोस देखे और कमेंट किया, लाइक किया, शेयर किया। इस वीडियो को भी प्लीज लाइक करना अगर आपको पसंद आए। बाकी अगर आपने चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाई।\n\nवर्किंग? तो शुड आई सजेस्ट यू बस का ट्रैवल और वह इस तरीके से रीअप्ट कर जाता है। ठीक है? तो दीज़ आर द मेन

In [37]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

## generate 

In [38]:
answer = llm.invoke(final_prompt)
print(answer)

content="I don't know." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 1060, 'total_tokens': 1064, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001614, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001614, 'upstream_inference_prompt_cost': 0.000159, 'upstream_inference_completions_cost': 2.4e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_32d32d2381', 'id': 'gen-1794203155-tWHy05Impiplzq9nSdag', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1bed7-7a9e-76b0-a201-4fe50119bf87-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1060, 'output_tokens': 4, 'total_token

## Building chains for the better workflow 

In [39]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [40]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [41]:
parallel_chain = RunnableParallel({
    'context':retrieval | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [42]:
parser = StrOutputParser()

In [43]:
main_chain = parallel_chain | prompt | llm | parser

In [45]:
interview_prompt = PromptTemplate(
    template="""
    You are an experienced technical interviewer.

    Analyze the following YouTube video transcript.

    Generate exactly 10 potential interview questions
    based on the concepts discussed in the transcript.

    Instructions:
    1. Use only concepts covered in the transcript.
    2. Include conceptual and technical questions.
    3. Include questions about important definitions,
       working principles and applications where relevant.
    4. Avoid duplicate questions.
    5. Arrange questions from basic to advanced.
    6. Do not invent topics that are not discussed.

    Transcript:
    {transcript}

    Output:
    Numbered list of exactly 10 interview questions.
    """,
    input_variables=["transcript"]
)

interview_chain = interview_prompt | llm | parser

In [46]:
interview_questions = interview_chain.invoke({
    "transcript": english_transcript
})

print(interview_questions)

1. **What is an AI agent, and how does it differ from traditional software applications?**

2. **Can you explain the problem that AI agents aim to solve, particularly in the context of planning a trip?**

3. **What are the main components of an AI agent as described in the transcript?**

4. **Define the term "React" in the context of AI agents. What does it stand for and what are its key characteristics?**

5. **How does context maintenance play a role in the operation of an AI agent?**

6. **What is the significance of using external tools and APIs in the functionality of an AI agent?**

7. **Describe the thought-action-observation loop in a React agent. How does it facilitate problem-solving?**

8. **What are the responsibilities of the agent executor in the context of an AI agent?**

9. **How does an AI agent utilize both a language model (LLM) and tools to execute tasks? Provide an example.**

10. **What are the limitations of using LangChain for developing scalable AI agents, and 

In [47]:
main_chain.invoke('What is an AI agent, and how does it differ from traditional software applications?')

'An AI agent is an intelligent system that possesses two main components: a large language model (LLM) and various tools. The AI agent uses the LLM for reasoning and decision-making, allowing it to process natural language and understand context. It also has access to APIs and databases, enabling it to utilize specific tools when necessary. \n\nThe key differences between an AI agent and traditional software applications include:\n\n1. **Reasoning Mechanism**: AI agents use LLMs for reasoning, which allows them to make decisions based on context and previous interactions.\n2. **Tool Utilization**: AI agents can access and utilize multiple tools (like APIs) to execute tasks, whereas traditional software may be more static and limited in functionality.\n3. **Context Awareness**: AI agents maintain a memory of their interactions and tasks, allowing them to adapt and adjust their plans based on new information.\n4. **Active Adjustment**: AI agents can dynamically re-plan and adjust their a